In [ ]:
#@title 1) آپلود زیرنویس و انتخاب ویدیو
DRIVE = False       #@param {type:"boolean"}
COMPRESS = True     #@param {type:"boolean"}
MAX_HEIGHT = 720    #@param {type:"integer"}

from google.colab import files
import os, subprocess, re

VID_EXT = ('.mp4', '.mkv', '.mov', '.webm', '.avi', '.m4v')

def parse_srt(path):
    raw = open(path, 'rb').read()
    for enc in ('utf-8-sig', 'utf-16', 'cp1256', 'latin-1'):
        try: txt = raw.decode(enc); break
        except Exception: continue
    txt = txt.replace('\r\n', '\n').replace('\r', '\n').strip()
    out = []
    for b in re.split(r'\n\s*\n', txt):
        lines = b.split('\n')
        idx = next((i for i, l in enumerate(lines) if '-->' in l), None)
        if idx is None: continue
        m = re.match(r'\s*(\d+):(\d+):(\d+)[,.](\d+)\s*-->\s*(\d+):(\d+):(\d+)[,.](\d+)', lines[idx])
        if not m: continue
        g = m.groups()
        t = lambda h, mi, s, ms: int(h) * 3600 + int(mi) * 60 + int(s) + float("0." + ms)
        text = ' '.join(lines[idx + 1:])
        text = re.sub(r'<[^>]+>|\{[^}]+\}', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            out.append({"s": t(*g[:4]), "e": t(*g[4:]), "t": text})
    return out

def prepare(src):
    # ویدیوی اصلی در VIDEO_PATH می‌ماند؛ نسخه‌ی سبک برای پلیر در video_path
    global VIDEO_PATH, video_path
    VIDEO_PATH = src
    video_path = src
    print("⏳ در حال خواندن ویدیو...")
    if COMPRESS:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src,
            "-vf", rf"scale=-2:min({MAX_HEIGHT}\,ih)", "-c:v", "libx264", "-crf", "28",
            "-preset", "veryfast", "-c:a", "aac", "-b:a", "96k", "-movflags", "+faststart",
            "/content/_play.mp4"], check=True)
        video_path = "/content/_play.mp4"
    mb = os.path.getsize(video_path) / 1e6
    print(f"📦 حجم ویدیوی پلیر: {mb:.1f} MB")
    if mb > 150:
        print("⚠️ ویدیو سنگین است؛ COMPRESS را روشن کن یا MAX_HEIGHT را کمتر کن.")
    print("✅ ویدیو آماده است. حالا سلول بعدی را اجرا کن.")

# ---- ۱) زیرنویس همیشه از حافظه‌ی دستگاه ----
print("📄 فایل زیرنویس (.srt) را از حافظه‌ی دستگاه انتخاب کن:")
up = files.upload()
srt_path = next((f for f in up if f.lower().endswith('.srt')), None)
assert srt_path, "فایل .srt آپلود نشد"
cues = parse_srt(srt_path)
print(f"✅ {len(cues)} دیالوگ خوانده شد.")

# ---- ۲) ویدیو: از دستگاه یا از گوگل درایو ----
if not DRIVE:
    print("🎬 حالا ویدیو را از حافظه‌ی دستگاه انتخاب کن:")
    up2 = files.upload()
    v = next((f for f in up2 if f.lower().endswith(VID_EXT)), None)
    assert v, "فایل ویدیو آپلود نشد"
    prepare(v)
else:
    import ipywidgets as widgets
    from google.colab import drive
    from IPython.display import display
    drive.mount('/content/drive')
    ROOT = "/content/drive/MyDrive"
    state = {"path": ROOT, "sel": None, "busy": False}
    title = widgets.HTML()
    lst = widgets.Select(options=[], rows=14, layout=widgets.Layout(width="100%"))
    btn = widgets.Button(description="✅ تأیید این ویدیو", button_style="success", disabled=True)
    out = widgets.Output()

    def refresh():
        p = state["path"]
        try: names = sorted(os.listdir(p), key=str.lower)
        except Exception: names = []
        items = []
        if p != ROOT: items.append(("⬆️ بازگشت", "up"))
        items += [("📁 " + n, "d:" + n) for n in names if not n.startswith('.') and os.path.isdir(os.path.join(p, n))]
        items += [("🎬 " + n, "f:" + n) for n in names if n.lower().endswith(VID_EXT)]
        state["busy"] = True
        lst.options = items
        lst.index = None
        state["busy"] = False
        sel = state["sel"]
        title.value = f"<b>📂 {p.replace('/content/drive/MyDrive', 'My Drive')}</b>" + \
                      (f"<br>انتخاب‌شده: <span style='color:#0a0'>{os.path.basename(sel)}</span>" if sel else "<br>یک ویدیو را انتخاب کن")

    def on_sel(change):
        v = change["new"]
        if state["busy"] or v is None: return
        if v == "up":
            state["path"] = os.path.dirname(state["path"]); refresh()
        elif v.startswith("d:"):
            state["path"] = os.path.join(state["path"], v[2:]); refresh()
        else:
            state["sel"] = os.path.join(state["path"], v[2:])
            btn.disabled = False
            refresh()

    def on_confirm(_):
        btn.disabled = True
        with out:
            prepare(state["sel"])

    btn.on_click(on_confirm)
    lst.observe(on_sel, 'value')
    refresh()
    print("📁 ویدیوی خودت را از گوگل درایو انتخاب کن (روی پوشه‌ها بزن تا باز شوند) و بعد «تأیید» را بزن:")
    display(widgets.VBox([title, lst, btn, out]))


In [ ]:
#@title 2) دقیق‌سازی زمان کلمات با صدا (اختیاری)
ALIGN = True        #@param {type:"boolean"}
ALIGN_LANG = "fas"  #@param {type:"string"}
HOLD = 0.15         # چند ثانیه کلمه‌ی آخر روی صفحه بماند

import subprocess, sys
assert "cues" in globals() and "VIDEO_PATH" in globals(), "اول سلول ۱ را کامل اجرا کن (در حالت درایو، ویدیو را هم تأیید کن)."

if ALIGN:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git"], check=True)
        import torch
        from ctc_forced_aligner import (load_audio, load_alignment_model, generate_emissions,
                                        preprocess_text, get_alignments, get_spans, postprocess_results)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO_PATH, "-vn", "-ac", "1",
                        "-ar", "16000", "/content/_audio.wav"], check=True)
        cuda = torch.cuda.is_available()
        if not cuda: print("⚠️ GPU فعال نیست؛ روی CPU کند است (Runtime ← Change runtime type ← GPU).")
        device = "cuda" if cuda else "cpu"
        model, tok = load_alignment_model(device, dtype=torch.float16 if cuda else torch.float32)
        wav = load_audio("/content/_audio.wav", model.dtype, model.device)
        print("⏳ در حال تحلیل صدا...")
        emissions, stride = generate_emissions(model, wav, batch_size=16 if cuda else 4)

        words_all = [w for c in cues for w in c["t"].split()]
        text = " ".join(words_all)
        res, err = None, None
        for lang in dict.fromkeys([ALIGN_LANG, "fas", "pes"]):
            try:
                res = preprocess_text(text, romanize=True, language=lang); break
            except Exception as e:
                err = e
        if res is None: raise RuntimeError(f"کد زبان پذیرفته نشد: {err}")
        tokens_starred, text_starred = res
        segments, scores, blank = get_alignments(emissions, tokens_starred, tok)
        spans = get_spans(tokens_starred, segments, blank)
        wts = postprocess_results(text_starred, spans, stride, scores)
        if len(wts) != len(words_all):
            raise RuntimeError(f"تعداد کلمات نخواند ({len(wts)} به‌جای {len(words_all)})")

        new, k, bad = [], 0, 0
        for c in cues:
            ws_ = c["t"].split(); n = len(ws_)
            w = [[a, float(p["start"]), float(p["end"])] for a, p in zip(ws_, wts[k:k + n])]
            k += n
            st_ = [x[1] for x in w]
            ok = all(b >= a for a, b in zip(st_, st_[1:])) and (n == 1 or st_[-1] - st_[0] >= 0.05 * (n - 1))
            if ok:
                new.append({"s": w[0][1], "e": w[-1][2], "t": c["t"], "w": w})
            else:   # زمان‌های این دیالوگ نامعتبر بود؛ همان تخمین قبلی
                bad += 1
                new.append({"s": c["s"], "e": c["e"], "t": c["t"]})
        for i, c in enumerate(new):
            nxt = new[i + 1]["s"] if i + 1 < len(new) else 1e9
            last = c["w"][-1][2] if c.get("w") else c["e"]
            c["e"] = max(min(c["e"] + HOLD, nxt), last)
        drift = [abs(a["s"] - b["s"]) for a, b in zip(new, cues)]
        print(f"✅ زمان کلمات از روی صدا محاسبه شد.")
        if bad: print(f"   ⚠️ برای {bad} از {len(new)} دیالوگ نتیجه نامعتبر بود و تخمین قبلی نگه داشته شد.")
        print(f"   اختلاف با زمان‌های SRT: میانگین {sum(drift)/len(drift):.2f}s ، بیشترین {max(drift):.2f}s")
        cues = new
    except Exception as e:
        print("⚠️ دقیق‌سازی ناموفق بود؛ زمان‌بندی تخمینی قبلی استفاده می‌شود.")
        print("   علت:", repr(e))
else:
    print("دقیق‌سازی خاموش است؛ از زمان‌بندی تخمینی استفاده می‌شود.")


In [ ]:
#@title 3) نمایش پلیر
from IPython.display import HTML, display
import base64, json
assert "cues" in globals() and "video_path" in globals(), "اول سلول ۱ را کامل اجرا کن (در حالت درایو، ویدیو را هم تأیید کن)."

# بررسی منطقی بودن زمان‌های SRT
import subprocess
try:
    _dur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", VIDEO_PATH]))
    _bad = [i + 1 for i, c in enumerate(cues)
            if c["s"] > _dur or (c["e"] - c["s"]) / max(1, len(c["t"].split())) > 3]
    if _bad:
        print(f"⚠️ زمان این دیالوگ‌ها در SRT غیرمنطقی است (خارج از طول ویدیو {_dur:.0f}s یا خیلی طولانی): {_bad[:15]}")
        print("   احتمالاً ستون دقیقه یا ثانیه اشتباه نوشته شده. سلول ۲ (دقیق‌سازی) را اجرا کن یا SRT را اصلاح کن.")
except Exception:
    pass

HTML_TEMPLATE = r"""<!DOCTYPE html><html><head><meta charset="utf-8">
<link href="https://fonts.googleapis.com/css2?family=Vazirmatn:wght@400;700;900&family=Lalezar&family=Noto+Naskh+Arabic:wght@500;700&family=Rubik:wght@500;800&family=Jomhuria&family=Gulzar&family=Oi&display=swap" rel="stylesheet">
<style>
*{box-sizing:border-box}
body{margin:0;font-family:Vazirmatn,Tahoma,sans-serif;background:#111;color:#eee}
#wrap{position:relative;width:100%;max-width:960px;margin:0 auto;background:#000;container-type:inline-size;overflow:hidden;
 --fs:5;--tc:#ffffff;--oc:#000000;--ow:4;--ac:#ffd400;--atc:#000000;--z:1.35;--font:Vazirmatn}
#vid{display:block;width:100%;height:auto;max-height:80vh;margin:0 auto;background:#000}
#caps{position:absolute;left:4%;right:4%;display:flex;flex-wrap:wrap;justify-content:center;align-items:center;
 gap:.15em .35em;pointer-events:none;font-family:var(--font),Tahoma,sans-serif;font-size:calc(var(--fs)*1cqw);
 line-height:1.6;text-align:center}
#caps.bottom{bottom:9%}#caps.middle{top:50%;transform:translateY(-50%)}#caps.top{top:6%}
.w{display:inline-block;padding:0 .22em;border-radius:.3em;color:var(--tc);font-weight:var(--fw,700);
 -webkit-text-stroke:calc(var(--ow)*.1cqw) var(--oc);paint-order:stroke fill;
 text-shadow:0 .05em .12em rgba(0,0,0,.45);transition:transform .12s cubic-bezier(.3,1.6,.5,1),background .1s;transform-origin:center}
.w.on{background:var(--ac);backdrop-filter:blur(var(--bl,0px));-webkit-backdrop-filter:blur(var(--bl,0px));border:1px solid var(--gb,transparent);color:var(--atc);-webkit-text-stroke:calc(var(--ow)*.1cqw) var(--oc2,transparent);transform:scale(var(--z));position:relative;z-index:2;
 box-shadow:0 .08em .3em rgba(0,0,0,.4)}
#wrap[data-mode="plain"] .w.on{transform:none !important}
#wrap[data-mode="glow"] .w.on{background:none;backdrop-filter:none;-webkit-backdrop-filter:none;border-color:transparent;box-shadow:none;transform:none;color:var(--acs);
 text-shadow:0 0 .12em var(--acg),0 0 .35em var(--acg),0 0 .8em var(--acg);animation:flash .35s ease-out}
@keyframes flash{0%{text-shadow:0 0 .2em #fff,0 0 .6em var(--acs),0 0 1.4em var(--acs)}100%{text-shadow:0 0 .12em var(--acg),0 0 .35em var(--acg),0 0 .8em var(--acg)}}
#wrap[data-mode="type"] .w.on{transform:none !important}
.w.hid{visibility:hidden}.w .h{visibility:hidden}
#bar{display:flex;gap:8px;justify-content:center;padding:8px}
button{background:#333;color:#fff;border:0;border-radius:8px;padding:6px 14px;cursor:pointer;font-family:inherit}
summary{max-width:960px;margin:6px auto;padding:8px 12px;background:#2b2b33;border-radius:10px;cursor:pointer;direction:rtl;font-weight:700}
details{max-width:960px;margin:0 auto}
#panel{margin:8px auto;padding:12px;display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:10px;
 background:#1d1d22;border-radius:12px;direction:rtl}
#panel label{display:flex;flex-direction:column;gap:4px;font-size:13px}
#panel input[type=color]{width:100%;height:32px;border:0;background:none;padding:0}
#panel select,#panel input[type=number]{padding:5px;border-radius:6px;border:1px solid #444;background:#2a2a30;color:#fff;font-family:inherit}
#wrap:fullscreen{max-width:none;display:flex;align-items:center;justify-content:center;height:100vh}
#wrap:fullscreen #vid{max-height:100vh;height:100%;object-fit:contain}
</style></head><body>
<details open id="det"><summary>⚙️ تنظیمات زیرنویس</summary>
<div id="panel">
 <label>حالت نمایش
  <select id="mode"><option value="zoom">کلمه‌ی فعال زوم شود 🔍</option><option value="plain">بدون زوم (فقط هایلایت)</option><option value="glow">درخشش نئونی ✨</option><option value="type">تایپ حرف‌به‌حرف ⌨️</option></select></label>
 <label>تعداد کلمه در هر صفحه (۰ = کل دیالوگ)
  <input id="n" type="number" min="0" max="30" value="3"></label>
 <label>شدت زوم <input id="zoom" type="range" min="1.05" max="2" step="0.05" value="1.35"></label>
 <label>اندازه فونت <input id="fs" type="range" min="2" max="10" step="0.1" value="5"></label>
 <label>ضخامت حاشیه <input id="ow" type="range" min="0" max="30" step="0.5" value="4"></label>
 <label>رنگ فونت <input id="tc" type="color" value="#ffffff"></label>
 <label>رنگ حاشیه <input id="oc" type="color" value="#000000"></label>
 <label>بک‌گراند کلمه‌ی فعال <input id="ac" type="color" value="#ffd400"></label>
 <label>وضوح بک‌گراند / شدت درخشش (کم = شیشه‌ای) <input id="ao" type="range" min="0.05" max="1" step="0.05" value="1"></label>
 <label>رنگ متن کلمه‌ی فعال <input id="atc" type="color" value="#000000"></label>
 <label>فونت
  <select id="font"><option>Vazirmatn</option><option>Lalezar</option><option>Noto Naskh Arabic</option><option>Rubik</option><option>Jomhuria</option><option>Gulzar</option><option>Oi</option><option>Tahoma</option></select></label>
 <label>موقعیت
  <select id="pos"><option value="bottom">پایین</option><option value="middle">وسط</option><option value="top">بالا</option></select></label>
</div>
</details>
<div id="wrap">
 <video id="vid" controls controlsList="nofullscreen" playsinline src="__VIDEO__"></video>
 <div id="caps" class="bottom"></div>
</div>
<div id="bar"><button id="fsb">⛶ تمام‌صفحه (با زیرنویس)</button></div>
<script>
const CUES = __CUES__;
const $ = id => document.getElementById(id);
const wrap=$('wrap'), vid=$('vid'), caps=$('caps');
let chunks=[], cur=-2, curW=-2, curK=-2;

function buildChunks(n){
  chunks=[];
  for(const c of CUES){
    let timed;
    if(c.w && c.w.length){
      timed=c.w.map(x=>({t:x[0],s:x[1],e:x[2]}));
      for(let i=0;i<timed.length-1;i++) timed[i].e=timed[i+1].s;
      timed[timed.length-1].e=c.e;
    } else {
      const ws=c.t.split(' ').filter(Boolean); if(!ws.length) continue;
      const wt=ws.map(w=>w.length+1), tot=wt.reduce((a,b)=>a+b,0), dur=Math.max(c.e-c.s,0.05);
      let t=c.s; timed=ws.map((w,i)=>{const d=dur*wt[i]/tot; const o={t:w,s:t,e:t+d}; t+=d; return o;});
    }
    const size = n>0 ? n : timed.length;
    for(let i=0;i<timed.length;i+=size){
      const part=timed.slice(i,i+size);
      chunks.push({s:part[0].s, e:(i+size>=timed.length)?c.e:part[part.length-1].e, w:part});
    }
  }
  cur=-2; curW=-2;
}
function applyStyle(){
  const st=wrap.style;
  st.setProperty('--fs',$('fs').value); st.setProperty('--ow',$('ow').value);
  st.setProperty('--tc',$('tc').value); st.setProperty('--oc',$('oc').value);
  const h=$('ac').value, a=parseFloat($('ao').value);
  st.setProperty('--ac','rgba('+parseInt(h.slice(1,3),16)+','+parseInt(h.slice(3,5),16)+','+parseInt(h.slice(5,7),16)+','+a+')');
  st.setProperty('--bl',((1-a)*10).toFixed(1)+'px');
  st.setProperty('--gb','rgba(255,255,255,'+((1-a)*0.55).toFixed(2)+')'); st.setProperty('--atc',$('atc').value);
  st.setProperty('--oc2',$('oc').value);
  st.setProperty('--acs',h);
  st.setProperty('--acg','rgba('+parseInt(h.slice(1,3),16)+','+parseInt(h.slice(3,5),16)+','+parseInt(h.slice(5,7),16)+','+a+')');
  curW=-2; curK=-2;
  st.setProperty('--z',$('zoom').value);
  wrap.dataset.mode=$('mode').value;
  $('zoom').disabled=($('mode').value!=='zoom');
  st.setProperty('--font',"'"+$('font').value+"'");
  caps.className=$('pos').value;
}
function findChunk(t){
  let lo=0, hi=chunks.length-1, r=-1;
  while(lo<=hi){const m=(lo+hi)>>1; if(chunks[m].s<=t){r=m;lo=m+1}else hi=m-1}
  return (r>=0 && t<chunks[r].e) ? r : -1;
}
function setWord(el,full,k){
  const ch=Array.from(full), v=el.firstChild, h=el.lastChild;
  if(k<0||k>=ch.length){v.textContent=full;h.textContent='';}
  else{v.textContent=ch.slice(0,k).join('');h.textContent=ch.slice(k).join('');}
}
function render(ci){
  caps.innerHTML='';
  if(ci<0) return;
  const ch=chunks[ci];
  caps.dir=/[\u0600-\u06FF]/.test(ch.w[0].t)?'rtl':'ltr';
  ch.w.forEach(w=>{const s=document.createElement('span'); s.className='w'; s.innerHTML='<span class="v"></span><span class="h"></span>'; s.firstChild.textContent=w.t; caps.appendChild(s);});
}
function tick(){
  const t=vid.currentTime, ci=findChunk(t);
  if(ci!==cur){cur=ci; curW=-2; curK=-2; render(ci);}
  if(ci>=0){
    const ws=chunks[ci].w; let wi=0;
    for(let i=0;i<ws.length;i++) if(ws[i].s<=t) wi=i;
    const typ=$('mode').value==='type';
    let k=-1;
    if(typ){
      const w=ws[wi], L=Array.from(w.t).length, dur=w.e-w.s, tt=Math.min(dur*0.7, L*0.06);
      k=(dur>0.15 && tt>0.01) ? Math.max(1,Math.min(L,Math.ceil(((t-w.s)/tt)*L))) : L;
    }
    if(wi!==curW||k!==curK){
      curW=wi; curK=k;
      [...caps.children].forEach((el,i)=>{
        el.classList.toggle('on',i===wi);
        el.classList.toggle('hid',typ&&i>wi);
        setWord(el,ws[i].t,(typ&&i===wi)?k:-1);
      });
    }
  }
  requestAnimationFrame(tick);
}
['fs','ow','tc','oc','ac','ao','atc','zoom','mode','font','pos'].forEach(id=>{$(id).addEventListener('input',applyStyle);$(id).addEventListener('change',applyStyle);});
$('n').addEventListener('input',()=>buildChunks(Math.max(0,parseInt($('n').value)||0)));
$('fsb').onclick=()=>document.fullscreenElement?document.exitFullscreen():wrap.requestFullscreen();
let _pt=null;
function getSettings(){return {fs:+$('fs').value,ow:+$('ow').value,tc:$('tc').value,oc:$('oc').value,ac:$('ac').value,ao:+$('ao').value,atc:$('atc').value,zoom:+$('zoom').value,mode:$('mode').value,font:$('font').value,pos:$('pos').value,n:Math.max(0,parseInt($('n').value)||0)};}
function pushSettings(){clearTimeout(_pt);_pt=setTimeout(()=>{try{google.colab.kernel.invokeFunction('save_caption_settings',[JSON.stringify(getSettings())],{});}catch(e){}},300);}
['fs','ow','tc','oc','ac','ao','atc','zoom','mode','font','pos','n'].forEach(id=>{$(id).addEventListener('input',pushSettings);$(id).addEventListener('change',pushSettings);});
applyStyle(); buildChunks(3); tick(); pushSettings();
</script></body></html>"""

mime = "video/webm" if video_path.lower().endswith(".webm") else "video/mp4"
with open(video_path, "rb") as f:
    vid_b64 = base64.b64encode(f.read()).decode()

page = (HTML_TEMPLATE
        .replace("__CUES__", json.dumps(cues, ensure_ascii=False).replace("</", "<\\/"))
        .replace("__VIDEO__", f"data:{mime};base64,{vid_b64}"))

from google.colab import output
caption_settings = {}
def _save_caption_settings(s):
    global caption_settings
    caption_settings = json.loads(s)
output.register_callback("save_caption_settings", _save_caption_settings)

display(HTML(page))



In [ ]:
#@title 4) رندر و دانلود ویدیوی نهایی با زیرنویس (با آخرین تنظیمات پلیر)
CRF = 15        #@param {type:"slider", min:10, max:23, step:1}
PRESET = "medium"  #@param ["veryfast","fast","medium","slow"]

import subprocess, json, os, math, re, sys, shutil, urllib.request
assert "cues" in globals() and "VIDEO_PATH" in globals(), "اول سلول ۱ را کامل اجرا کن."
from functools import lru_cache
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
subprocess.run("apt-get -qq install -y libraqm0 libfribidi0 libharfbuzz0b > /dev/null 2>&1", shell=True)
import arabic_reshaper
try:
    from bidi.algorithm import get_display
except Exception:
    from bidi import get_display
from PIL import Image, ImageDraw, ImageFont, ImageFilter, features
from google.colab import files
RAQM = features.check("raqm")
print("موتور متن فارسی:", "Raqm (شکل‌دهی داخلی)" if RAQM else "arabic-reshaper")

DEFAULTS = dict(fs=5, ow=4, tc="#ffffff", oc="#000000", ac="#ffd400", ao=1.0,
                atc="#000000", zoom=1.35, mode="zoom", font="Vazirmatn", pos="bottom", n=3)
st = dict(DEFAULTS)
if globals().get("caption_settings"):
    st.update(caption_settings)
else:
    print("⚠️ تنظیمات پلیر دریافت نشد؛ از تنظیمات پیش‌فرض استفاده می‌شود. (سلول ۲ را اجرا کن و یک بار یکی از تنظیمات را تکان بده)")
print("🎛 تنظیمات استفاده‌شده:", st)

SRC = VIDEO_PATH   # ویدیوی اصلی (نه نسخه‌ی فشرده‌شده)
probe = json.loads(subprocess.check_output(["ffprobe","-v","error","-select_streams","v:0",
    "-show_entries","stream=width,height,r_frame_rate:stream_tags=rotate:stream_side_data=rotation",
    "-of","json",SRC]))
s0 = probe["streams"][0]
W, H = int(s0["width"]), int(s0["height"])
rot = int(s0.get("tags", {}).get("rotate", 0) or 0)
for sd in s0.get("side_data_list", []):
    if "rotation" in sd: rot = int(sd["rotation"])
if abs(rot) % 180 == 90: W, H = H, W
_n, _d = s0["r_frame_rate"].split("/")
FPS = float(_n) / (float(_d) or 1)
print(f"🎞 ویدیو: {W}x{H} @ {FPS:.2f}fps")

# ---------- فونت ----------
GF = "https://github.com/google/fonts/raw/main/ofl/"
FONT_URLS = {
    "Vazirmatn": ["https://github.com/rastikerdar/vazirmatn/raw/master/fonts/ttf/Vazirmatn-Bold.ttf",
                  GF + "vazirmatn/Vazirmatn%5Bwght%5D.ttf"],
    "Lalezar": [GF + "lalezar/Lalezar-Regular.ttf"],
    "Noto Naskh Arabic": [GF + "notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf"],
    "Rubik": [GF + "rubik/Rubik%5Bwght%5D.ttf"],
    "Jomhuria": [GF + "jomhuria/Jomhuria-Regular.ttf"],
    "Gulzar": [GF + "gulzar/Gulzar-Regular.ttf"],
    "Oi": [GF + "oi/Oi-Regular.ttf"],
}
os.makedirs("/content/fonts", exist_ok=True)
def get_font_path(name):
    for nm in (name, "Vazirmatn"):
        p = f"/content/fonts/{nm.replace(' ', '_')}.ttf"
        if os.path.exists(p): return p
        for url in FONT_URLS.get(nm, []):
            try:
                urllib.request.urlretrieve(url, p)
                if os.path.getsize(p) > 10000: return p
            except Exception:
                pass
        if nm != "Vazirmatn": print(f"⚠️ فونت {nm} دانلود نشد؛ از وزیرمتن استفاده می‌شود.")
    raise RuntimeError("دانلود فونت ناموفق بود (اینترنت Colab را بررسی کن).")
FONT_PATH = get_font_path(st["font"])

@lru_cache(maxsize=None)
def load_font(size):
    f = ImageFont.truetype(FONT_PATH, max(1, int(round(size))))
    try: f.set_variation_by_axes([700])
    except Exception: pass
    return f

def hex2rgb(h): return tuple(int(h[i:i+2], 16) for i in (1, 3, 5))
AR = "[\u0600-\u06FF]"
def prep(txt, rtl):
    # با Raqm خود Pillow شکل‌دهی و راست‌به‌چپ را انجام می‌دهد (نباید دوباره reshape شود)
    if RAQM:
        r = bool(re.search(AR, txt))
        return txt, ({"direction": "rtl", "language": "fa"} if r else {"direction": "ltr"})
    if rtl:
        txt = txt.replace("\u200c", "")
        return get_display(arabic_reshaper.reshape(txt)), {}
    return txt, {}

BASE = st["fs"] * W / 100.0                 # اندازه فونت (مثل cqw در پلیر)
SW = st["ow"] * W / 1000.0 / 2.0            # ضخامت بیرونی حاشیه
SS = 4                                      # سوپرسمپل برای لبه‌های نرم

MODE = st["mode"]
GLOW = MODE == "glow"
TYPE = MODE == "type"

def margin(size, active):
    return int(math.ceil(SW)) + 4 + (int(.55 * size) if (GLOW and active) else 0)

@lru_cache(maxsize=None)
def word_img(text, size, active, rtl, shown=None, flash=False):
    f = load_font(size)
    t_full, kw_full = prep(text, rtl)
    t, kw = prep(shown, rtl) if shown else (t_full, kw_full)
    pad = .22 * size
    bw, bh = f.getlength(t_full, **kw_full) + 2 * pad, 1.6 * size
    m = margin(size, active)
    w, h = int(math.ceil(bw)) + 2 * m, int(math.ceil(bh)) + 2 * m
    img = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    def colorize(mask, rgb):
        c = Image.new("RGBA", (w, h), rgb + (0,)); c.putalpha(mask); return c
    if active and not GLOW:
        a = float(st["ao"])
        box = [m * SS, m * SS, (m + bw) * SS, (m + bh) * SS]
        mk = Image.new("L", (w * SS, h * SS), 0)
        ImageDraw.Draw(mk).rounded_rectangle(box, radius=.3 * size * SS, fill=255)
        mk = mk.resize((w, h), Image.LANCZOS).point(lambda v: int(v * a))
        img = Image.alpha_composite(img, colorize(mk, hex2rgb(st["ac"])))
        if a < 1:
            rk = Image.new("L", (w * SS, h * SS), 0)
            ImageDraw.Draw(rk).rounded_rectangle(box, radius=.3 * size * SS, outline=255, width=max(SS, int(size * SS / 50)))
            rk = rk.resize((w, h), Image.LANCZOS).point(lambda v: int(v * (1 - a) * .55))
            img = Image.alpha_composite(img, colorize(rk, (255, 255, 255)))
    word_rtl = (kw.get("direction") == "rtl") if RAQM else rtl
    xy, anc = ((m + bw - pad, m + bh / 2), "rm") if word_rtl else ((m + pad, m + bh / 2), "lm")
    sw_i = int(round(SW))
    fill_m = Image.new("L", (w, h), 0)
    ImageDraw.Draw(fill_m).text(xy, t, font=f, anchor=anc, fill=255, **kw)
    str_m = None
    if SW > 0 or (active and GLOW):
        str_m = Image.new("L", (w, h), 0)
        ImageDraw.Draw(str_m).text(xy, t, font=f, anchor=anc, fill=255, stroke_width=sw_i, stroke_fill=255, **kw)
    if active and GLOW:
        k = float(st["ao"]) * (1.0 if flash else 0.7)
        for rad, wgt in ((.08 * size, 1.0), (.25 * size, 0.9)):
            g = str_m.filter(ImageFilter.GaussianBlur(rad)).point(lambda v, wgt=wgt: min(255, int(v * wgt * k * 2)))
            img = Image.alpha_composite(img, colorize(g, hex2rgb(st["ac"])))
    if SW > 0:
        img = Image.alpha_composite(img, colorize(str_m, hex2rgb(st["oc"])))
    if active and GLOW:
        tc = hex2rgb(st["ac"])
    else:
        tc = hex2rgb(st["atc"] if active else st["tc"])
    return Image.alpha_composite(img, colorize(fill_m, tc))

def layout(words, rtl):
    f = load_font(BASE)
    pad, gap = .22 * BASE, .35 * BASE
    widths = [f.getlength(prep(w, rtl)[0], **prep(w, rtl)[1]) + 2 * pad for w in words]
    maxw = W * 0.92
    lines, cur, curw = [], [], 0
    for i, wd in enumerate(widths):
        add = wd + (gap if cur else 0)
        if cur and curw + add > maxw:
            lines.append(cur); cur, curw = [], 0; add = wd
        cur.append(i); curw += add
    if cur: lines.append(cur)
    lh, rg = 1.6 * BASE, .15 * BASE
    bh = len(lines) * lh + (len(lines) - 1) * rg
    top = {"bottom": H * 0.91 - bh, "middle": H / 2 - bh / 2, "top": H * 0.06}[st["pos"]]
    pos = [None] * len(words)
    for li, ln in enumerate(lines):
        lw = sum(widths[i] for i in ln) + gap * (len(ln) - 1)
        cy = top + li * (lh + rg) + lh / 2
        if rtl:
            x = W / 2 + lw / 2
            for i in ln:
                pos[i] = (x - widths[i] / 2, cy); x -= widths[i] + gap
        else:
            x = W / 2 - lw / 2
            for i in ln:
                pos[i] = (x + widths[i] / 2, cy); x += widths[i] + gap
    return pos

def render_frame(words, rtl, ai, scale, typed=None, flash=False):
    frame = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    for i, (wd, (cx, cy)) in enumerate(zip(words, layout(words, rtl))):
        if TYPE and i > ai:
            continue                              # کلمه‌های بعدی هنوز تایپ نشده‌اند
        if i == ai:
            sh = None
            if TYPE and typed is not None and typed < len(wd):
                sh = wd[:typed]
            img = word_img(wd, round(BASE * scale, 2), True, rtl, sh, flash)
        else:
            img = word_img(wd, round(BASE, 2), False, rtl)
        x = int(round(cx - img.width / 2)); y = int(round(cy - img.height / 2))
        x = min(max(0, x), max(0, W - img.width)); y = min(max(0, y), max(0, H - img.height))
        frame.alpha_composite(img, (x, y))
    return frame

# ---------- زمان‌بندی (همان منطق پلیر) ----------
def build_chunks(n):
    out = []
    for c in cues:
        if c.get("w"):
            timed = [(w[0], w[1], w[2]) for w in c["w"]]
            timed = [(timed[i][0], timed[i][1], timed[i + 1][1]) for i in range(len(timed) - 1)] + [(timed[-1][0], timed[-1][1], c["e"])]
        else:
            ws = c["t"].split()
            if not ws: continue
            wt = [len(w) + 1 for w in ws]; tot = sum(wt); dur = max(c["e"] - c["s"], 0.05)
            t = c["s"]; timed = []
            for w, k in zip(ws, wt):
                d = dur * k / tot; timed.append((w, t, t + d)); t += d
        size = n if n > 0 else len(timed)
        for i in range(0, len(timed), size):
            part = timed[i:i + size]
            out.append({"s": part[0][1], "e": c["e"] if i + size >= len(timed) else part[-1][2], "w": part})
    return out

chunks = build_chunks(int(st["n"]))
zoom_on = st["mode"] == "zoom"
Z = float(st["zoom"]) if zoom_on else 1.0

shutil.rmtree("/content/ovl", ignore_errors=True); os.makedirs("/content/ovl")
blank = "/content/ovl/blank.png"
Image.new("RGBA", (W, H), (0, 0, 0, 0)).save(blank)
segs, t, idx = [], 0.0, 0
def add_seg(a, b, frame):
    global idx
    if b - a < 1e-3: return
    p = f"/content/ovl/{idx:06d}.png"; idx += 1
    frame.save(p, compress_level=1); segs.append((p, b - a))

print("🖌 در حال ساخت فریم‌های زیرنویس...")
for ci, ch in enumerate(chunks):
    if ch["s"] > t + 1e-3:
        segs.append((blank, ch["s"] - t)); t = ch["s"]
    words = [w[0] for w in ch["w"]]
    rtl = bool(re.search("[\u0600-\u06FF]", words[0]))
    for wi, (_, ws, we) in enumerate(ch["w"]):
        ws = max(ws, t)
        if we <= ws + 1e-3: continue
        dur = we - ws
        if TYPE and dur > 0.15:
            L = len(words[wi]); tt = min(dur * 0.7, L * 0.06); dt = tt / L
            for k in range(1, L + 1):
                a0 = ws + (k - 1) * dt
                b0 = we if k == L else ws + k * dt
                add_seg(a0, b0, render_frame(words, rtl, wi, 1.0, typed=k))
        elif zoom_on and dur > 0.25:
            pop = min(0.08, dur / 3)
            add_seg(ws, ws + pop, render_frame(words, rtl, wi, (1 + Z) / 2))
            add_seg(ws + pop, we, render_frame(words, rtl, wi, Z))
        elif GLOW and dur > 0.25:
            pop = min(0.1, dur / 3)
            add_seg(ws, ws + pop, render_frame(words, rtl, wi, 1.0, flash=True))
            add_seg(ws + pop, we, render_frame(words, rtl, wi, 1.0))
        else:
            add_seg(ws, we, render_frame(words, rtl, wi, Z))
        t = we
    if ci % 100 == 0: print(f"   {ci}/{len(chunks)}")

with open("/content/ovl/list.txt", "w") as f:
    for p, d in segs:
        f.write(f"file '{p}'\nduration {d:.5f}\n")
    f.write(f"file '{segs[-1][0]}'\n")

# ---------- رندر نهایی ----------
OUT = "/content/captioned_video.mp4"
fc = f"[1:v]fps={FPS},format=rgba[ov];[0:v][ov]overlay=0:0:eof_action=pass:format=auto,format=yuv420p[v]"
def run(audio):
    cmd = ["ffmpeg","-y","-hide_banner","-loglevel","error","-stats","-i",SRC,"-f","concat","-safe","0",
           "-i","/content/ovl/list.txt","-filter_complex",fc,"-map","[v]","-map","0:a?",
           "-c:v","libx264","-preset",PRESET,"-crf",str(CRF)] + audio + ["-movflags","+faststart",OUT]
    return subprocess.run(cmd).returncode
print("🎬 در حال رندر نهایی (بسته به طول ویدیو چند دقیقه طول می‌کشد)...")
if run(["-c:a","copy"]) != 0:
    print("صدا قابل کپی مستقیم نبود؛ با AAC باکیفیت تبدیل می‌شود...")
    assert run(["-c:a","aac","-b:a","256k"]) == 0, "رندر ناموفق بود"
print(f"✅ تمام شد. حجم خروجی: {os.path.getsize(OUT)/1e6:.1f} MB")
from IPython.display import HTML, display
display(HTML("""
<div style="text-align:center;margin:18px 0;font-family:sans-serif">
 <a href="https://www.youtube.com/@aigolden" target="_blank" rel="noopener"
    style="display:inline-block;background:#ff0000;color:#fff;text-decoration:none;font-weight:700;
           padding:12px 28px;border-radius:999px;font-size:16px;box-shadow:0 4px 14px rgba(255,0,0,.35)">
   ▶ دنبال کردن در یوتیوب
 </a>
</div>"""))
files.download(OUT)
